In [ ]:
import subprocess, sys

_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "physlink==0.1.2"],
    capture_output=True, text=True
)
if _result.returncode != 0:
    print(_result.stdout)
    raise RuntimeError(
        "physlink could not be installed — check the version number or PyPI availability"
    )
print(_result.stdout[-500:] if _result.stdout else "physlink installed successfully.")

In [ ]:
from physlink import ObservationSpace, ActionSpace, DreamerV3Adapter, register_invariant

In [ ]:
# ⚠️ Edit this cell — this is the ONLY cell you need to modify
#
# Replace your_trajectories with your simulation data.
# Each entry is a dict. Include the fields your invariant reads.
# (obs and action are required by the adapter — use zeros if your data lacks them.)

import numpy as np

_rng = np.random.default_rng(42)  # fixed seed for idempotent example data

your_trajectories = [
    {
        "obs": _rng.random(14).tolist(),        # observation vector (14-dim for 7 joints + velocities)
        "action": _rng.random(7).tolist(),       # action vector (7-dim)
        "mass_flow_in": 1.0 + _rng.normal(0, 0.003),    # ← your physics field
        "mass_flow_out": 1.0 + _rng.normal(0, 0.003),   # ← your physics field
    }
    for _ in range(1000)
]
print(f"Loaded {len(your_trajectories)} trajectories.")

In [ ]:
obs_space = ObservationSpace.from_proprioception(joints=7, include_velocity=True)
act_space = ActionSpace.continuous(dims=7, bounds=[(-1.0, 1.0)] * 7)

In [ ]:
from physlink import DreamerV3Adapter

# Fresh adapter instance ensures full idempotence when re-running this cell
adapter = DreamerV3Adapter(obs_space, act_space)

def mass_conservation(trajectory: dict) -> float:
    """Returns residual: absolute difference between mass_flow_in and mass_flow_out."""
    return abs(trajectory["mass_flow_in"] - trajectory["mass_flow_out"])

register_invariant(
    adapter,
    name="mass_conservation",
    fn=mass_conservation,
    tolerance=0.01,
    mode="hard",
)

# steps=100 is sufficient for compliance validation (not performance training)
adapter.fit(your_trajectories, steps=100, checkpoint_interval_steps=50)

report = adapter.compliance_report()
print(report.summary())

In [ ]:
report.plot(title="Mass Conservation Check", show_threshold=True)

In [ ]:
adapter.export(path="./physlink_export/")
print("Export complete. Files saved to ./physlink_export/")

## What's next?

Your domain validation is complete. Here are your next steps:

1. **Share your results**: Your compliance report and GIF are in `./physlink_export/` — share the notebook URL with your team.

2. **Add your domain to PhysLink**: If you want to contribute your invariant (`mass_conservation` or your own) to the PhysLink community, open a Domain Extension issue:

   [**Open a Domain Extension Issue →**](https://github.com/YOUR-ORG/physlink/issues/new?template=domain_extension.md)

   The template will guide you through describing your physical domain, invariant function, and expected PASS output.

3. **Evaluate for your lab**: See the [Lab Adoption Guide](../docs/lab-adoption-guide.md) for a structured 1-day evaluation checklist.